# AI-Powered Predictive Maintenance System for Smart Manufacturing

# 1 ) Import Required Libraries


In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

# 2 ) Data Collection & Loading

Datasets Used
1. Telemetry Data
2. Error Logs
3. Maintenance Records
4. Failure Records
5. Machine Information

In [3]:
telemetry = pd.read_csv("PdM_telemetry.csv")
errors = pd.read_csv("PdM_errors.csv")
maint = pd.read_csv("PdM_maint.csv")
failures = pd.read_csv("PdM_failures.csv")
machines = pd.read_csv("PdM_machines.csv")

# 3 ) Dataset Overview


In [4]:
datasets = {
    "Telemetry": telemetry,
    "Errors": errors,
    "Maintenance": maint,
    "Failures": failures,
    "Machines": machines
}

overview = pd.DataFrame({
    "Dataset": [name for name in datasets.keys()],
    "Rows": [df.shape[0] for df in datasets.values()],
    "Columns": [df.shape[1] for df in datasets.values()],
    "Missing Values": [df.isnull().sum().sum() for df in datasets.values()],
    "Duplicate Rows": [df.duplicated().sum() for df in datasets.values()]
})

overview

,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,Telemetry,876100,6,0,0
1,Errors,3919,3,0,0
2,Maintenance,3286,3,0,0
3,Failures,761,3,0,0
4,Machines,100,3,0,0


# 4 ) Data Preprocessing

## 4.1 Datetime Conversion
*  The datasets contain timestamps stored as text values. To perform accurate time-based operations and merging, all datetime columns must be converted into a standardized datetime format.

In [47]:
# Convert datetime columns into pandas datetime format

telemetry['datetime'] = pd.to_datetime(telemetry['datetime'])
errors['datetime'] = pd.to_datetime(errors['datetime'])
maint['datetime'] = pd.to_datetime(maint['datetime'])
failures['datetime'] = pd.to_datetime(failures['datetime'])

* Consistent datetime format across all datasets.
* Accurate merging and time-based analysis.

## 4.2 Duplicate Record Validation
* Duplicate records can introduce bias and incorrect patterns during model training. Therefore, duplicate observations must be identified and removed before merging.

In [48]:
# Check duplicate records in each dataset

print("Telemetry :", telemetry.duplicated().sum())
print("Errors :", errors.duplicated().sum())
print("Maintenance :", maint.duplicated().sum())
print("Failures :", failures.duplicated().sum())
print("Machines :", machines.duplicated().sum())

Telemetry : 0
Errors : 0
Maintenance : 0
Failures : 0
Machines : 0


Remove Duplicates

In [49]:
# Remove duplicate records

telemetry.drop_duplicates(inplace=True)
errors.drop_duplicates(inplace=True)
maint.drop_duplicates(inplace=True)
failures.drop_duplicates(inplace=True)
machines.drop_duplicates(inplace=True)

* Improved data quality.
* Prevention of duplicate machine observations.

## 4.3 Missing Value Check

In [50]:
# Check missing values

print(telemetry.isnull().sum())
print(errors.isnull().sum())
print(maint.isnull().sum())
print(failures.isnull().sum())
print(machines.isnull().sum())

datetime     0
machineID    0
volt         0
rotate       0
pressure     0
vibration    0
dtype: int64
datetime     0
machineID    0
errorID      0
dtype: int64
datetime     0
machineID    0
comp         0
dtype: int64
datetime        0
machineID       0
failure         0
failure_flag    0
dtype: int64
machineID    0
model        0
age          0
dtype: int64


* Identify whether any preprocessing is required before integration.


# 5 ) Data Integration & Merging
Objective

* The objective of this phase is to combine machine telemetry, maintenance history, error logs, machine information, and failure records into a single consolidated dataset.

## 5.1 Create Base Dataset
* Telemetry data contains continuous machine sensor readings and acts as the foundation of the final dataset.

In [51]:
# Create base dataset

merged_df = telemetry.copy()

print("Base Shape:", merged_df.shape)

Base Shape: (876100, 6)


* Telemetry becomes the primary dataset because it contains continuous machine operating information.

## 5.2 Merge Machine Information
Machine age and model information provide important context about machine characteristics and failure behavior.

In [52]:
# Merge machine metadata

merged_df = merged_df.merge(
    machines,
    on='machineID',
    how='left'
)

* Each telemetry record now includes machine model and age information.

# 5.3 Transform Error Records
* Machine learning algorithms require numerical features. Error categories must therefore be converted into numerical indicators

In [53]:
# Convert error categories into numerical features

errors_encoded = pd.get_dummies(
    errors,
    columns=['errorID']
)

* Error categories are transformed into binary features.

# 5.4 Aggregate Error Records
* Multiple error events may occur at the same timestamp. Aggregation prevents row duplication during merging.

In [55]:
# Aggregate error records
# One row per machineID + datetime

errors_encoded = (
    errors_encoded
    .groupby(
        ['machineID', 'datetime'],
        as_index=False
    )
    .sum()
)
# Check duplicate keys

errors_encoded[
    ['machineID','datetime']
].duplicated().sum()

np.int64(0)

* One unique record is maintained per machine and timestamp.

## 5.5 Merge Error Information
* Machine errors often indicate abnormal machine behavior and are important predictors of failure.

In [56]:
# Merge error features

merged_df = merged_df.merge(
    errors_encoded,
    on=['machineID','datetime'],
    how='left'
)

* Error history is now attached to telemetry observations.

## 5.6 Transform Maintenance Records

* Maintenance events provide valuable information about machine servicing and component replacement.

In [57]:
# Convert component categories into numerical features

maint_encoded = pd.get_dummies(
    maint,
    columns=['comp']
)

* Maintenance categories become model-friendly numerical features.

## 5.7 Aggregate Maintenance Records
* Aggregation ensures that multiple maintenance actions occurring at the same timestamp do not create duplicate observations.

In [59]:
# Aggregate maintenance events

maint_encoded = (
    maint_encoded
    .groupby(
        ['machineID','datetime'],
        as_index=False
    )
    .sum()
)

# validate
maint_encoded[
    ['machineID','datetime']
].duplicated().sum()

np.int64(0)

* One maintenance record exists for each machine and timestamp.

## 5.8 Merge Maintenance Information
* Maintenance history is often strongly related to machine health and failure probability.

In [60]:
# Merge maintenance features

merged_df = merged_df.merge(
    maint_encoded,
    on=['machineID','datetime'],
    how='left'
)

* Machine maintenance history is successfully integrated.

## 5.9 Create Failure Label
* Failure records represent the target variable that the machine learning model will learn to predict.

In [63]:
# Create binary target

failures['failure_flag'] = 1

# Aggregate failures

failure_df = (
    failures[
        ['machineID','datetime','failure_flag']
    ]
    .groupby(
        ['machineID','datetime'],
        as_index=False
    )
    .max()
)

# validate
failure_df[
    ['machineID','datetime']
].duplicated().sum()

np.int64(0)

* Failure events are represented using binary labels.

## 5.10 Merge Failure Information
* Failure labels must be attached to machine observations to support supervised learning.

In [64]:
# Merge failure labels

merged_df = merged_df.merge(
    failure_df,
    on=['machineID','datetime'],
    how='left'
)

* Each machine observation now contains failure information.

## 5.11 Handle Missing Values
* Missing values in error, maintenance, and failure columns simply indicate that no event occurred.

In [65]:
# Replace missing values with 0

merged_df.fillna(0, inplace=True)

* No Error = 0
* No Maintenance = 0
* No Failure = 0

## 5.12 Merge Validation
* This step verifies that the merging process has not introduced overlapping records or duplicate observations.

In [66]:
# Check duplicate rows

print(
    "Duplicate Rows:",
    merged_df.duplicated().sum()
)

# Check unique machineID-datetime pairs

print(
    "Duplicate Keys:",
    merged_df[
        ['machineID','datetime']
    ].duplicated().sum()
)

# Check missing values

print(
    "Missing Values:",
    merged_df.isnull().sum().sum()
)

Duplicate Rows: 0
Duplicate Keys: 0
Missing Values: 0


## 5.13 Export Merged Dataset
* Save the cleaned and validated merged dataset for future feature engineering and modeling tasks.

In [69]:
# Save merged dataset

merged_df.to_csv(
    "merged_dataset.csv",
    index=False
)

print("Merged dataset saved successfully.")

Merged dataset saved successfully.


In [70]:
# Check dataset shape before saving

print("Dataset Shape:", merged_df.shape)

Dataset Shape: (876100, 18)


In [71]:
# Display all column names

for col in merged_df.columns:
    print(col)

datetime
machineID
volt
rotate
pressure
vibration
model
age
errorID_error1
errorID_error2
errorID_error3
errorID_error4
errorID_error5
comp_comp1
comp_comp2
comp_comp3
comp_comp4
failure_flag


In [72]:
# Dataset information

merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 876100 entries, 0 to 876099
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   datetime        876100 non-null  datetime64[ns]
 1   machineID       876100 non-null  int64         
 2   volt            876100 non-null  float64       
 3   rotate          876100 non-null  float64       
 4   pressure        876100 non-null  float64       
 5   vibration       876100 non-null  float64       
 6   model           876100 non-null  object        
 7   age             876100 non-null  int64         
 8   errorID_error1  876100 non-null  float64       
 9   errorID_error2  876100 non-null  float64       
 10  errorID_error3  876100 non-null  float64       
 11  errorID_error4  876100 non-null  float64       
 12  errorID_error5  876100 non-null  float64       
 13  comp_comp1      876100 non-null  float64       
 14  comp_comp2      876100 non-null  flo

In [73]:
# Identify constant columns

constant_cols = []

for col in merged_df.columns:

    if merged_df[col].nunique() == 1:

        constant_cols.append(col)

print("Constant Columns:")
print(constant_cols)

Constant Columns:
[]


In [74]:
unique_df = pd.DataFrame({

    "Feature": merged_df.columns,

    "Unique Values": [
        merged_df[col].nunique()
        for col in merged_df.columns
    ]

})

unique_df.sort_values(
    by="Unique Values"
)

,Feature,Unique Values
15,comp_comp3,2
14,comp_comp2,2
13,comp_comp1,2
12,errorID_error5,2
11,errorID_error4,2
10,errorID_error3,2
9,errorID_error2,2
8,errorID_error1,2
17,failure_flag,2
16,comp_comp4,2
